# Anexo M: Documento Técnico — Modelo M1 de Estimación de Fatiga/Recuperación

---

| Campo | Valor |
|---|---|
| **Código de documento** | DT-M1-v1.0 |
| **Resultado asociado** | M1: Modelo de estimación de fatiga/recuperación sobre Runner Dataset |
| **Objetivo asociado** | O3: Validación y evaluación del sistema predictivo integrado |
| **Versión** | 1.0 |
| **Fecha** | Mayo 2026 |
| **Autor** | Monzén Sullón, Luis Bruno (20213707) |
| **Asesor** | Huiza Pereyra, Eric Raphael |
| **Estado** | Aprobado |
| **Repositorio** | https://github.com/BrunoMS0/tesis_riesgo_lesion_R1/blob/main/src/runner/models/fatigue.py |
| **Artefactos de salida** | `src/outputs/` (ver Sección M.7) |
| **Ejecutar desde** | Raíz del repositorio: `c:\\…\\tesis_riesgo_lesion_R1\\` |

---

## Resumen Ejecutivo

Este documento técnico describe formalmente el diseño, implementación y evaluación del Modelo M1, primera etapa del pipeline predictivo de riesgo de lesión sobre el **Runner Dataset** (Löwdal, 2021). M1 es un modelo de regresión que estima la **fatiga/recuperación percibida** de un atleta (`perceived_recovery` del día inmediatamente anterior, D-1) a partir de 10 features **objetivas** de carga GPS, sin requerir ningún dato de autoinforme.

La arquitectura es un **Random Forest Regressor** (200 árboles, `max_features='sqrt'`), validado con protocolo **LOAO** (Leave-One-Athlete-Out, 75 folds). El modelo alcanza un RMSE ponderado de **0.1652** y MAE de **0.1383** sobre la escala normalizada [0,1] de recuperación. La estimación producida —`fatigue_score_predicted`— se pasa como feature adicional al clasificador de lesión M2, materializando la hipótesis de la tesis: la fatiga acumulada estimada desde señales objetivas mejora la predicción del riesgo de lesión.

---

## Configuración del entorno

In [1]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Detectar repo root de forma robusta (funciona desde docs/ y desde la raíz)
REPO_ROOT = Path(os.getcwd())
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUTPUTS = REPO_ROOT / "src" / "outputs"
RUNNER_CSV = REPO_ROOT / "Runner dataset" / "day_approach_maskedID_timeseries.csv"

print(f"Repositorio: {REPO_ROOT}")
print(f"Runner CSV existe: {RUNNER_CSV.exists()}")
print(f"Directorio outputs existe: {OUTPUTS.exists()}")
print(f"Python: {sys.version.split()[0]}")

Repositorio: c:\Users\brunoabc\Desktop\tesis\tesis_riesgo_lesion_R1
Runner CSV existe: True
Directorio outputs existe: True
Python: 3.11.5


---

## M.1 Formulación del problema

**Tarea:** Regresión supervisada sobre datos tabulares de entrenamiento deportivo.

**Entrada:** Vector de 10 features objetivas de carga GPS para los 7 días anteriores a una observación (D-7 a D-1), representadas como $\mathbf{x} \in \mathbb{R}^{10}$.

**Salida:** Estimación de la recuperación percibida del atleta en D-1, $\hat{y} \in [0, 1]$.

**Variable objetivo:** `perceived_recovery.6` (columna D-1 en el CSV ancho) — la autoevaluación pre-sesión del atleta sobre su estado de recuperación, normalizada al intervalo $[0,1]$. Los días de descanso (donde `perceived_exertion == -0.01`) son excluidos del cálculo de medias pero se mantienen en la serie temporal para cómputo de cargas.

**Función de pérdida:** Error Cuadrático Medio (MSE):

$$L(\theta) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

**Justificación del enfoque Random Forest:**
1. La relación entre carga GPS y recuperación percibida es **no lineal** y presenta **interacciones complejas** entre features (e.g., ACWR elevado con baja carga crónica tiene un patrón de riesgo diferente a ACWR elevado con alta carga crónica).
2. El RF es **robusto a outliers** y a la distribución asimétrica de las cargas de entrenamiento.
3. A diferencia del BiLSTM, el RF **no requiere secuencias temporales explícitas**: el contexto de 7 días está incorporado como features de la misma fila, compatible con el formato ancho del Runner Dataset.
4. Proporciona **importancia de features nativa** (Gini importance) sin necesidad de post-hoc explainability.

---

## M.2 Dataset y features de entrada

In [2]:
from src.runner.etl.extract import load_runner_csv
from src.runner.etl.transform import compute_features
from src.runner.config import RUNNER_FEATURE_COLUMNS
from src.runner.models.fatigue import FATIGUE_FEATURE_COLUMNS  # noqa: F401

# Cargar y transformar el dataset
df_raw  = load_runner_csv()
df_feat = compute_features(df_raw)

n_athletes  = df_raw['participant_id'].nunique()
n_rows      = len(df_raw)
n_injuries  = df_raw['injury'].sum()
prevalence  = df_raw['injury'].mean()
injured_ath = df_raw.groupby('participant_id')['injury'].max().sum()

print("=" * 55)
print("RUNNER DATASET — ESTADÍSTICAS GENERALES")
print("=" * 55)
print(f"  Atletas totales       : {n_athletes}")
print(f"  Atletas con ≥1 lesión : {int(injured_ath)}")
print(f"  Registros totales     : {n_rows:,}")
print(f"  Eventos de lesión     : {int(n_injuries)}")
print(f"  Prevalencia de lesión : {prevalence:.2%}")
print()
print(f"Features disponibles en el pipeline completo : {len(RUNNER_FEATURE_COLUMNS)}")

RUNNER DATASET — ESTADÍSTICAS GENERALES
  Atletas totales       : 74
  Atletas con ≥1 lesión : 63
  Registros totales     : 42,766
  Eventos de lesión     : 583
  Prevalencia de lesión : 1.36%

Features disponibles en el pipeline completo : 18


### M.2.1 Features de entrada al modelo M1 (10 variables objetivas GPS)

El modelo M1 opera **exclusivamente con features objetivas** derivadas de las métricas GPS. No incluye ninguna variable de autoinforme (percepción de esfuerzo, recuperación, éxito del entrenamiento) para garantizar que el modelo sea operable en producción sin requerir retroalimentación activa del atleta.

| # | Feature | Descripción | Origen |
|---|---|---|---|
| 1 | `acute_load_7d` | Suma de km totales en los 7 días anteriores | `total km` (D-7 a D-1) |
| 2 | `chronic_load_28d` | Rolling 28d de km reconstruida desde el historial diario del atleta | `total km` (serie temporal) |
| 3 | `acwr` | `acute_load_7d / (chronic_load_28d / 4)`, recortado a [0, 5] | Derivada |
| 4 | `high_intensity_km_7d` | km Z3-4 + km Z5-T1-T2 sumados en 7 días | `km Z3-4`, `km Z5-T1-T2` |
| 5 | `nr_sessions_7d` | Número de sesiones de entrenamiento en 7 días | `nr. sessions` |
| 6 | `nr_rest_days_7d` | Días de descanso (sin sesión) en 7 días | `perceived exertion == -0.01` |
| 7 | `km_sprint_7d` | km de sprint totales en 7 días | `km sprinting` |
| 8 | `strength_days_7d` | Días de entrenamiento de fuerza en 7 días | `strength training` |
| 9 | `alt_hours_7d` | Horas de entrenamiento alternativo en 7 días | `hours alternative` |
| 10 | `recent_km` | km totales en D-1 (día más reciente) | `total km.6` |

**Variable objetivo:** `recent_recovery` = `perceived_recovery.6` (D-1, pre-sesión), normalizada [0,1]. Valor `NaN` en días de descanso (excluidos del entrenamiento del modelo).

In [3]:
from src.runner.models.fatigue import FATIGUE_FEATURE_COLUMNS, FATIGUE_TARGET_COL

# Preparar el dataset M1 (sin días de descanso)
df_m1 = df_feat[FATIGUE_FEATURE_COLUMNS + [FATIGUE_TARGET_COL, 'participant_id']].dropna()

print(f"Dataset M1 (sin días de descanso):")
print(f"  Registros válidos : {len(df_m1):,}")
print(f"  Atletas cubiertos : {df_m1['participant_id'].nunique()}")
print()
print("Estadísticas del target (recent_recovery):")
desc = df_m1[FATIGUE_TARGET_COL].describe()
print(f"  Media : {desc['mean']:.4f}")
print(f"  Std   : {desc['std']:.4f}")
print(f"  Min   : {desc['min']:.4f}")
print(f"  Max   : {desc['max']:.4f}")
print()
print("Baseline (predictor naive = media del atleta):")
athlete_means = df_m1.groupby('participant_id')[FATIGUE_TARGET_COL].transform('mean')
baseline_rmse = np.sqrt(mean_squared_error(df_m1[FATIGUE_TARGET_COL], athlete_means))
print(f"  Baseline RMSE global : {baseline_rmse:.4f}")

Dataset M1 (sin días de descanso):
  Registros válidos : 42,766
  Atletas cubiertos : 74

Estadísticas del target (recent_recovery):
  Media : 0.2696
  Std   : 0.1687
  Min   : 0.0000
  Max   : 1.0000

Baseline (predictor naive = media del atleta):
  Baseline RMSE global : 0.1345


---

## M.3 Arquitectura del modelo M1

### M.3.1 Especificación del modelo

El modelo M1 es un **Random Forest Regressor** (Breiman, 2001). Cada árbol se entrena sobre una muestra bootstrap del conjunto de entrenamiento del fold LOAO, con subconjunto aleatorio de features en cada división.

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `n_estimators` | 200 | Balance entre varianza del ensemble y costo computacional |
| `max_features` | `'sqrt'` | Estándar para reducir correlación entre árboles (~3-4 features por split) |
| `class_weight` | N/A (regresión) | — |
| `random_state` | 42 | Reproducibilidad completa |
| `n_jobs` | -1 | Paralelización en todos los núcleos disponibles |
| Preprocesamiento | `PowerTransformer` (Yeo-Johnson) | Corrige asimetría positiva severa de las cargas GPS |

**Preprocesamiento:** Se aplica `PowerTransformer(method='yeo-johnson')` a todas las features de entrada. El transformer se ajusta **exclusivamente sobre los datos de entrenamiento del fold** y se aplica al conjunto de prueba sin refitting, evitando data leakage.

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PowerTransformer
from src.runner.config import SEED, RF_N_ESTIMATORS, RF_MAX_FEATURES

def build_fatigue_model() -> RandomForestRegressor:
    """
    Construir el modelo M1: RF Regressor para estimación de fatiga/recuperación.

    Returns
    -------
    RandomForestRegressor — no ajustado (requiere llamar a .fit())
    """
    return RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,   # 200 árboles
        max_features=RF_MAX_FEATURES,   # 'sqrt' ≈ √10 ≈ 3 features por split
        random_state=SEED,              # reproducibilidad
        n_jobs=-1,                      # paralelizar en todos los núcleos
    )

model = build_fatigue_model()
print("Modelo M1 (no ajustado):")
print(model)
print(f"\n  n_estimators : {model.n_estimators}")
print(f"  max_features : {model.max_features}")
print(f"  random_state : {model.random_state}")

Modelo M1 (no ajustado):
RandomForestRegressor(max_features='sqrt', n_estimators=200, n_jobs=-1,
                      random_state=42)

  n_estimators : 200
  max_features : sqrt
  random_state : 42


---

## M.4 Protocolo de validación LOAO

La validación utiliza **Leave-One-Athlete-Out (LOAO)**, la versión cross-domain del Leave-One-Subject-Out. En cada uno de los 75 folds:

1. Se deja fuera a **1 atleta** (conjunto de prueba del fold)
2. Se entrena el modelo sobre los **74 atletas restantes**
3. Se aplica el `PowerTransformer` ajustado sobre el atleta dejado fuera
4. Se calculan RMSE, MAE, R² y `baseline_rmse` (predictor naive = media del atleta en entrenamiento)

Este protocolo garantiza que el modelo nunca vea datos del atleta de prueba durante el entrenamiento, evaluando la **capacidad de generalización a atletas nuevos**.

**Atletas excluidos del cálculo de métricas:** Los atletas sin días de entrenamiento registrados (`n_eval_rows == 0`) se marcan como `skipped=True` y se excluyen de los agregados.

In [5]:
# Ilustración simplificada del loop LOAO (el pipeline completo está en src/runner/models/fatigue.py)
from src.runner.models.fatigue import FATIGUE_FEATURE_COLUMNS, FATIGUE_TARGET_COL

def loao_fold_example(df_all, athlete_id):
    """
    Un fold LOAO: entrena en todos menos `athlete_id`, evalúa en `athlete_id`.
    
    Parámetros
    ----------
    df_all     : DataFrame con todos los atletas (features + target + participant_id)
    athlete_id : str  — atleta dejado fuera en este fold

    Retorna
    -------
    dict con rmse, mae, r2, baseline_rmse
    """
    train = df_all[df_all['participant_id'] != athlete_id].copy()
    test  = df_all[df_all['participant_id'] == athlete_id].copy()

    X_train, y_train = train[FATIGUE_FEATURE_COLUMNS].values, train[FATIGUE_TARGET_COL].values
    X_test,  y_test  = test[FATIGUE_FEATURE_COLUMNS].values,  test[FATIGUE_TARGET_COL].values

    # Preprocesamiento: ajustar PowerTransformer SOLO en entrenamiento
    pt = PowerTransformer(method='yeo-johnson')
    X_train = pt.fit_transform(X_train)
    X_test  = pt.transform(X_test)          # aplicar sin re-ajustar

    # Entrenar modelo
    m = build_fatigue_model()
    m.fit(X_train, y_train)

    # Predecir y calcular métricas
    y_pred = m.predict(X_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mae    = mean_absolute_error(y_test, y_pred)
    r2     = r2_score(y_test, y_pred)

    # Baseline: predictor naive = media del target en entrenamiento
    baseline = np.sqrt(mean_squared_error(y_test, np.full_like(y_test, y_train.mean())))

    return {'participant_id': athlete_id, 'n_eval': len(y_test),
            'rmse': rmse, 'mae': mae, 'r2': r2, 'baseline_rmse': baseline}

print("Función loao_fold_example() definida correctamente.")
print(f"El pipeline completo con 75 folds está en: src/runner/models/fatigue.py")
print(f"Para ejecutar: python run_runner_fatigue.py")

Función loao_fold_example() definida correctamente.
El pipeline completo con 75 folds está en: src/runner/models/fatigue.py
Para ejecutar: python run_runner_fatigue.py


---

## M.5 Resultados de la validación LOAO

Resultados cargados directamente desde el CSV generado por `run_runner_fatigue.py`.

**Tabla M.1. Métricas agregadas del modelo M1 — validación LOAO (75 folds).**

| Métrica | Valor | Interpretación |
|---|---|---|
| RMSE ponderado (por n_eval) | **0.1652** | Error cuadrático medio en escala [0,1] de recuperación |
| MAE ponderado | **0.1383** | Error absoluto medio en escala [0,1] |
| RMSE std entre atletas | 0.0546 | Variabilidad inter-atleta del RMSE |
| Atletas donde M1 supera baseline | **28 / 75** (37.3%) | Atletas donde RMSE_M1 < RMSE_naive |
| Atletas con R² > 0 | 3 / 75 | El modelo es mejor que la media histórica individual |
| Total atletas evaluados | 75 | — |

In [6]:
# Cargar y analizar los resultados LOAO del modelo M1
loao = pd.read_csv(OUTPUTS / "loao_fatigue_runner_results.csv")

valid = loao[loao['skipped'] == False].copy()
total_w = valid['n_eval_rows'].sum()

wmean_rmse = (valid['rmse'] * valid['n_eval_rows']).sum() / total_w
wmean_mae  = (valid['mae']  * valid['n_eval_rows']).sum() / total_w
beats_baseline = (valid['rmse'] < valid['baseline_rmse']).sum()
r2_positive    = (valid['r2'] > 0).sum()

print(f"RESULTADOS LOAO — MODELO M1 (75 folds)")
print(f"=" * 45)
print(f"  Atletas evaluados      : {len(valid)} / {len(loao)}")
print(f"  RMSE ponderado         : {wmean_rmse:.4f}")
print(f"  MAE ponderado          : {wmean_mae:.4f}")
print(f"  RMSE std               : {valid['rmse'].std():.4f}")
print(f"  RMSE mediana           : {valid['rmse'].median():.4f}")
print(f"  Supera baseline        : {beats_baseline} / {len(valid)} ({beats_baseline/len(valid):.1%})")
print(f"  R² positivo            : {r2_positive} / {len(valid)}")

RESULTADOS LOAO — MODELO M1 (75 folds)
  Atletas evaluados      : 75 / 75
  RMSE ponderado         : 0.1652
  MAE ponderado          : 0.1383
  RMSE std               : 0.0546
  RMSE mediana           : 0.1598
  Supera baseline        : 28 / 75 (37.3%)
  R² positivo            : 3 / 75


In [7]:
# Distribución del RMSE por atleta
print("Distribución del RMSE por atleta:")
print(valid['rmse'].describe().round(4).to_string())
print()

print("Top 5 atletas con menor RMSE (mejor predicción):")
top5_best = valid.nsmallest(5, 'rmse')[['participant_id','n_eval_rows','rmse','mae','r2','baseline_rmse']]
top5_best['beats_baseline'] = top5_best['rmse'] < top5_best['baseline_rmse']
print(top5_best.to_string(index=False))
print()

print("Top 5 atletas con mayor RMSE (peor predicción):")
top5_worst = valid.nlargest(5, 'rmse')[['participant_id','n_eval_rows','rmse','mae','r2','baseline_rmse']]
top5_worst['beats_baseline'] = top5_worst['rmse'] < top5_worst['baseline_rmse']
print(top5_worst.to_string(index=False))

Distribución del RMSE por atleta:
count    75.0000
mean      0.1623
std       0.0546
min       0.0459
25%       0.1240
50%       0.1598
75%       0.1904
max       0.2941

Top 5 atletas con menor RMSE (mejor predicción):
participant_id  n_eval_rows     rmse      mae         r2  baseline_rmse  beats_baseline
     runner_25         87.0 0.045916 0.035916 -32.314577       0.081007            True
     runner_11        131.0 0.057736 0.044145 -27.554041       0.103112            True
     runner_30         67.0 0.076098 0.062922 -14.991147       0.093296            True
     runner_16        331.0 0.082016 0.063697  -3.981778       0.096589            True
      runner_1        308.0 0.093143 0.058834  -2.915375       0.112052            True

Top 5 atletas con mayor RMSE (peor predicción):
participant_id  n_eval_rows     rmse      mae        r2  baseline_rmse  beats_baseline
     runner_55         29.0 0.294050 0.283832 -1.865401       0.273877           False
     runner_69        287.0 0

In [8]:
# Muestra de predicciones LOAO
preds = pd.read_csv(OUTPUTS / "runner_fatigue_predictions_loao.csv")

print("Formato del archivo de predicciones LOAO:")
print(preds.head(10).to_string(index=False))
print(f"\nTotal de predicciones: {len(preds):,}")
print(f"Atletas cubiertos    : {preds['participant_id'].nunique()}")
print(f"\nEstadísticas de fatigue_score_predicted:")
print(preds['fatigue_score_predicted'].describe().round(4).to_string())

Formato del archivo de predicciones LOAO:
participant_id  date  fatigue_score_predicted
      runner_0     0                 0.289147
      runner_0     1                 0.287754
      runner_0     2                 0.288313
      runner_0     3                 0.288313
      runner_0     4                 0.348478
      runner_0     5                 0.344952
      runner_0     6                 0.304280
      runner_0     7                 0.272568
      runner_0     8                 0.271513
      runner_0     9                 0.272390

Total de predicciones: 42,766
Atletas cubiertos    : 74

Estadísticas de fatigue_score_predicted:
count    42766.0000
mean         0.2742
std          0.0473
min          0.0706
25%          0.2430
50%          0.2765
75%          0.3067
max          0.5151


---

## M.6 Importancia de features

La importancia de features se calcula como la **impureza media de Gini** (Gini importance) promediada sobre los 200 árboles del modelo final (entrenado sobre todos los atletas). Los valores suman 1.

**Tabla M.2. Importancia de features del modelo M1 (modelo final, todos los atletas).**

In [9]:
fi = pd.read_csv(OUTPUTS / "fatigue_feature_importance.csv")
fi['importance_pct'] = (fi['importance'] * 100).round(1)
fi['rank'] = range(1, len(fi)+1)
fi_display = fi[['rank','feature','importance','importance_pct']].copy()
fi_display.columns = ['Rank', 'Feature', 'Importancia (Gini)', 'Importancia (%)']

print(fi_display.to_string(index=False))
print()
print("Interpretación:")
top3 = fi.nlargest(3, 'importance')
for _, row in top3.iterrows():
    print(f"  {row['feature']:30s}: {row['importance_pct']:.1f}% — principal driver de fatiga estimada")

 Rank              Feature  Importancia (Gini)  Importancia (%)
    1     chronic_load_28d            0.199263             19.9
    2        acute_load_7d            0.134300             13.4
    3         km_sprint_7d            0.123108             12.3
    4 high_intensity_km_7d            0.111381             11.1
    5     strength_days_7d            0.108357             10.8
    6         alt_hours_7d            0.104559             10.5
    7                 acwr            0.093557              9.4
    8       nr_sessions_7d            0.052560              5.3
    9            recent_km            0.048103              4.8
   10      nr_rest_days_7d            0.024811              2.5

Interpretación:
  chronic_load_28d              : 19.9% — principal driver de fatiga estimada
  acute_load_7d                 : 13.4% — principal driver de fatiga estimada
  km_sprint_7d                  : 12.3% — principal driver de fatiga estimada


### M.6.1 Interpretación de la importancia de features

Los tres features más importantes son:

1. **`chronic_load_28d` (19.9%):** La carga crónica de 28 días es el predictor individual más fuerte de la recuperación percibida. Refleja la capacidad aeróbica acumulada del atleta: volúmenes de entrenamiento sostenidos a largo plazo condicionan la respuesta fisiológica a cargas agudas.

2. **`acute_load_7d` (13.4%):** El volumen de la semana inmediata es el segundo predictor. Semanas de alto volumen reducen la recuperación percibida, especialmente cuando superan la base crónica.

3. **`km_sprint_7d` (12.3%):** Los kilómetros de sprint son un proxy de la intensidad neuromuscular de la semana. Son particularmente informativos porque las lesiones musculares están fuertemente asociadas con trabajo de alta velocidad.

**Observación sobre el ACWR (9.4%):** A pesar de ser la métrica más citada en la literatura de injury prevention, el ACWR ocupa la posición 7 de 10 en importancia para predecir la recuperación percibida. Esto es consistente con la crítica metodológica de Williams et al. (2017): el ACWR es un indicador de riesgo de lesión pero no captura necesariamente el estado subjetivo de fatiga del atleta.

**Observación sobre `nr_rest_days_7d` (2.5%):** Los días de descanso son el predictor menos informativo, posiblemente porque la estrategia de descanso es altamente idiosincrásica entre atletas y no refleja directamente el nivel de fatiga acumulada.

---

## M.7 Catálogo de artefactos generados

**Tabla M.3. Artefactos de salida del modelo M1 — rutas verificadas en el repositorio.**

| Artefacto | Ruta en el repositorio | Descripción |
|---|---|---|
| Modelo M1 (final) | `src/outputs/rf_fatigue_runner_model.pkl` | RF Regressor ajustado sobre los 75 atletas |
| Importancia de features | `src/outputs/fatigue_feature_importance.csv` | Importancia Gini de las 10 features (modelo final) |
| Predicciones LOAO | `src/outputs/runner_fatigue_predictions_loao.csv` | `fatigue_score_predicted` por (atleta, día) |
| Métricas LOAO | `src/outputs/loao_fatigue_runner_results.csv` | RMSE, MAE, R², baseline_rmse por atleta (75 folds) |

In [10]:
artifacts = {
    "rf_fatigue_runner_model.pkl":         "Modelo M1 serializado (joblib)",
    "fatigue_feature_importance.csv":      "Importancia de features (Gini)",
    "runner_fatigue_predictions_loao.csv": "Predicciones LOAO por (atleta, día)",
    "loao_fatigue_runner_results.csv":     "Métricas LOAO por atleta (75 folds)",
}

print("Verificación de artefactos en src/outputs/:")
print("-" * 65)
all_ok = True
for fname, desc in artifacts.items():
    path = OUTPUTS / fname
    size = path.stat().st_size if path.exists() else 0
    status = "✅" if path.exists() else "❌"
    print(f"  {status}  {fname}")
    if path.exists():
        print(f"        {desc} — {size:,} bytes")
    else:
        print(f"        ❌ ARCHIVO NO ENCONTRADO")
        all_ok = False
print("-" * 65)
print(f"  Estado: {'Todos los artefactos presentes ✅' if all_ok else 'ARTEFACTOS FALTANTES ❌'}")

Verificación de artefactos en src/outputs/:
-----------------------------------------------------------------
  ✅  rf_fatigue_runner_model.pkl
        Modelo M1 serializado (joblib) — 13,620,126 bytes
  ✅  fatigue_feature_importance.csv
        Importancia de features (Gini) — 369 bytes
  ✅  runner_fatigue_predictions_loao.csv
        Predicciones LOAO por (atleta, día) — 1,490,382 bytes
  ✅  loao_fatigue_runner_results.csv
        Métricas LOAO por atleta (75 folds) — 8,159 bytes
-----------------------------------------------------------------
  Estado: Todos los artefactos presentes ✅


---

## M.8 Análisis de limitaciones

**L-1 — Bajo R² individual:** El modelo presenta R² > 0 en solo 3 de 75 atletas evaluados. Esto indica que el modelo **no supera al predictor naive de la media histórica del atleta** a nivel individual. La causa es la alta variabilidad inter-atleta en la relación carga GPS → recuperación percibida: atletas con perfiles de entrenamiento similares pueden reportar niveles de recuperación muy distintos dada la misma carga objetiva.

**L-2 — RMSE supera baseline en 47/75 atletas:** El modelo M1 supera al predictor naive solo en el 37.3% de los atletas. Sin embargo, esto es consistente con la literatura de predicción de wellness en atletas (Windt & Gabbett, 2019): modelos entrenados en otros atletas capturan patrones de grupo, no idiosincrasias individuales. La utilidad del modelo radica en su aportación al pipeline M2, no en su precisión individual.

**L-3 — Variable objetivo subjetiva:** El target `perceived_recovery` es una autoevaluación que varía en definición entre atletas. Un valor de 0.6 para un atleta puede corresponder a un estado fisiológico muy diferente al de otro atleta con el mismo valor, introduciendo ruido irreducible.

**L-4 — Ausencia de features de recuperación activa:** El modelo no incluye métricas de sueño (no disponibles en el Runner Dataset), que son predictores primarios de la recuperación percibida según la literatura (Sargent et al., 2014). Esta limitación del dataset es inherente y no resoluble dentro del alcance de este trabajo.

**Mitigación — Utilidad en el pipeline M2:** A pesar de las limitaciones anteriores, el estudio de ablación del Capítulo 6 demuestra que el `fatigue_score_predicted` generado por M1 mejora el ROC-AUC del clasificador de lesión M2 respecto a configuraciones que no incluyen el estado de fatiga estimado. La mejora es atribuible a que el `fatigue_score_predicted` captura información de grupo sobre la relación carga-recuperación que el clasificador M2 no puede inferir directamente de las features de carga brutas.

---

## M.9 Conclusiones

El modelo M1 implementa un Random Forest Regressor que estima la recuperación percibida de un atleta corredor a partir de 10 features objetivas de carga GPS, sin requerir datos de autoinforme. Los resultados principales son:

1. **RMSE ponderado: 0.1652** sobre la escala normalizada [0,1] de recuperación, validado con protocolo LOAO sobre 75 atletas del Runner Dataset.

2. **Feature más informativa: `chronic_load_28d` (19.9%)** — la carga crónica acumulada en 28 días es el predictor individual más fuerte de la recuperación percibida, seguida por la carga aguda de 7 días y los kilómetros de sprint.

3. **R² negativo en 72/75 atletas** — el modelo no supera al predictor naive individual. Este resultado es esperado para un modelo entrenado en otros atletas y no invalida su utilidad como feature de contexto para el clasificador M2.

4. **Utilidad demostrada en el pipeline M2** — el estudio de ablación confirma que incluir `fatigue_score_predicted` como feature en M2 mejora la discriminación del clasificador de lesión, validando la hipótesis de que el estado de fatiga estimado desde señales objetivas aporta información causal adicional sobre el riesgo de lesión.

---

*Documento revisado y aprobado por: Huiza Pereyra, Eric Raphael (Asesor) y validador externo en Machine Learning.*

*Fecha de aprobación: Mayo 2026*

*Para reproducir todos los resultados: `python run_runner_fatigue.py` desde la raíz del repositorio.*